<a href="https://colab.research.google.com/github/ANNYM25/python-together/blob/main/08-machine-learning/08_cnn_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8 · Convolutional neural networks (CNNs) — seeing images

A regular network treats an image as a flat list of pixels and throws away the
fact that nearby pixels belong together. **CNNs** fix that with a *convolution*:
a tiny window (a "filter") slides across the image looking for a pattern — an
edge, a corner, a texture. Stack these and the network learns shapes, then
objects. We'll do the core operation by hand in NumPy so it's not magic.

In [ ]:
import numpy as np

## A tiny "image"

An image is just a grid of numbers (brightness). Here's an 6×6 image with a
bright square sitting in a dark background.

In [ ]:
img = np.zeros((6, 6))
img[1:5, 1:5] = 1.0          # a bright block
print(img)

## A filter (kernel) that detects vertical edges

This 3×3 filter responds where brightness changes left-to-right. Sliding it over
the image and summing `patch * filter` at each spot is a **convolution**.

In [ ]:
vertical_edge = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1],
])

def convolve2d(image, kernel):
    kh, kw = kernel.shape
    out_h = image.shape[0] - kh + 1
    out_w = image.shape[1] - kw + 1
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = image[i:i + kh, j:j + kw]
            out[i, j] = np.sum(patch * kernel)       # the convolution step
    return out

feature_map = convolve2d(img, vertical_edge)
print("feature map (big values = a vertical edge was found):")
print(feature_map.round(1))

Notice the strong responses line up with the **left and right sides** of the
bright block — exactly where vertical edges live. A CNN doesn't hand-design this
filter; it **learns** the filter numbers by gradient descent, the same way our
lesson-7 net learned its weights.

## ReLU + pooling, the other two CNN ingredients

- **ReLU**: keep positive responses, zero the rest (same activation as lesson 7).
- **Max-pooling**: shrink the map by keeping the strongest value in each little
  window — gives a bit of "where-ish, don't care about exact pixel" robustness.

In [ ]:
relu_map = np.maximum(0, feature_map)

def max_pool(x, size=2):
    h, w = x.shape[0] // size, x.shape[1] // size
    return np.array([[x[i*size:(i+1)*size, j*size:(j+1)*size].max()
                      for j in range(w)] for i in range(h)])

print("after ReLU + 2x2 max-pool:")
print(max_pool(relu_map).round(1))

**A CNN is just:** convolution → ReLU → pooling, repeated, then a normal neural
net (lesson 7) on top to make the final decision. That's how models read MNIST
digits, X-rays, and satellite photos.

## The real thing (PyTorch) — the shape your NumPy maps onto

You wouldn't hand-roll convolutions in production. With PyTorch installed
(`pip install torch`) the same idea is a few lines — shown here as a comment so
this lesson still runs without the heavy dependency:

```python
import torch.nn as nn
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3),   # 16 learnable filters like ours
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(16 * 2 * 2, 10),         # 10 classes, e.g. digits 0-9
)
```

## 🏋️ Your turn

Make a **horizontal** edge detector (transpose the filter) and convolve it with
`img`. Where do the strong responses appear now — top and bottom instead of left
and right?

In [ ]:
horizontal_edge = vertical_edge.T
# print(convolve2d(img, horizontal_edge).round(1))

### ✅ What you learned
- An image is a grid of numbers; CNNs respect its 2-D structure.
- A **convolution** slides a small filter to detect local patterns (edges, etc.).
- CNNs **learn** their filters; ReLU + pooling add non-linearity and robustness.
- Stack conv→relu→pool, finish with a dense net — that's an image classifier.

**Next:** `09_kaggle_workflow.py` — put it all together on a real-style dataset.